> ⚠️ **作業中 (Work in Progress)**:このノートブックは現在開発中です。一部のコードが不完全であったり変更される可能性があります。

## 📋 目次

- [環境準備](#環境準備)
- [Resource Groupの作成](#resource-groupの作成)
- [Foundryリソースの作成](#foundryリソースの作成)
- [リソースの確認](#リソースの確認)
- [次のステップ](#次のステップ)

## 🎯 学習目標

- Azure CLIでResource Groupを作成
- Azure CLIまたはBicepでMicrosoft Foundryリソースを作成
- コードベースのInfrastructure as Code (IaC)実習

## ⏱️ 予想所要時間

約10分

## 環境準備

### 必須要件

1. **Azure CLIのインストール確認**
  - 以下のセルを実行して、Azure CLIがインストールされているか確認します。
  - インストールされていない場合:[Azure CLIインストールガイド](https://learn.microsoft.com/cli/azure/install-azure-cli)

2. **Azureログイン**
  - Azure CLIでAzureアカウントにログインします。

## Python仮想環境の設定

仮想環境を使用すると、プロジェクトごとに独立したPythonパッケージを管理できます。

### 仮想環境とは？

仮想環境（Virtual Environment）は、プロジェクトごとに独立したPython実行環境を作成します。

**メリット:**
- プロジェクトごとに異なるバージョンのパッケージを使用可能
- システムPython環境をクリーンに維持
- チームメンバー間で同じ開発環境を共有しやすい

### 仮想環境の作成と有効化

**1. ターミナルで仮想環境を作成**

macOS / Linux:
```bash
python3 -m venv .venv
```

Windows:
```bash
python -m venv .venv
```

**2. 仮想環境の有効化**

macOS / Linux:
```bash
source .venv/bin/activate
```

Windows (PowerShell):
```powershell
.venv\Scripts\Activate.ps1
```

Windows (CMD):
```cmd
.venv\Scripts\activate.bat
```

**3. 有効化の確認**

有効化されると、ターミナルプロンプトの前に`(.venv)`が表示されます:
```
(.venv) user@machine:~/project$
```

### VS Code Jupyter使用時

VS CodeでJupyter Notebookを使用する場合:
1. 仮想環境作成後
2. 右上の**カーネル選択**ボタンをクリック
3. **'.venv'環境**を選択

以降、すべてのセルは自動的に仮想環境で実行されます。

In [ ]:
# 必須パッケージインストール
# 仮想環境が有効化された状態で実行してください

!pip install -q azure-ai-projects azure-identity azure-mgmt-resource

print("✅ 必須パッケージのインストール完了!")
print("\n📦 インストールされたパッケージ:")
print("  - azure-ai-projects")
print("  - azure-identity")
print("  - azure-mgmt-resource")
print("\n💡 Azure CLIはシステムレベルで別途インストールが必要です:")
print("  https://learn.microsoft.com/cli/azure/install-azure-cli")

In [ ]:
# PATH 環境変数の設定
# JupyterカーネルでAzure CLIを見つけられるようにパスを追加
import os
import subprocess

# Azure CLIがインストールされる可能性のある複数のパス確認
possible_paths = [
  "/opt/homebrew/bin", # macOS (Apple Silicon)
  "/usr/local/bin",   # macOS (Intel) / Linux
  "/usr/bin",      # Linux / GitHub Codespaces
  "/home/linuxbrew/.linuxbrew/bin" # Linux Homebrew
]

# az コマンドパス検索
az_path = None
try:
  result = subprocess.run(['which', 'az'], capture_output=True, text=True)
  if result.returncode == 0:
    az_path = os.path.dirname(result.stdout.strip())
    print(f"🔍 Azure CLI 発見:{result.stdout.strip()}")
except:
  pass

# 発見されたパスまたは可能なパスたちを PATHに追加
paths_to_add = []
if az_path and az_path not in os.environ.get("PATH", ""):
  paths_to_add.append(az_path)
else:
  # whichで見つけないできなかった場合可能なパスたち追加
  for path in possible_paths:
    if os.path.exists(path) and path not in os.environ.get("PATH", ""):
      paths_to_add.append(path)

if paths_to_add:
  new_path = ":".join(paths_to_add) + ":" + os.environ.get("PATH", "")
  os.environ["PATH"] = new_path
  print(f"✅ PATHに追加されたパス:{', '.join(paths_to_add)}")
else:
  print("✅ PATHがが未正しく設定されてあります.")

print(f"\n💡 現在 PATH (最初 150者):{os.environ['PATH'][:150]}...")

In [ ]:
# Azure 認証 (Python SDK 使用)
from azure.identity import InteractiveBrowserCredential, DeviceCodeCredential
from azure.mgmt.resource import SubscriptionClient
import os

# テナント ID 設定 (必要時変更)
TENANT_ID = os.getenv("AZURE_TENANT_ID", "16b3c013-d300-468d-ac64-7eda0820b6d3")

# GitHub Codespaces またはリモート環境検出
IS_CODESPACES = os.getenv("CODESPACES") == "true" or os.getenv("CODESPACE_NAME") is not None
IS_REMOTE = os.getenv("REMOTE_CONTAINERS") == "true" or IS_CODESPACES

print("🔐 Azure 認証開始...")
print(f"Tenant ID:{TENANT_ID}")

if IS_REMOTE:
  print("\n💡 リモート環境(Codespaces/Remote Container)が検出されました.")
  print("デバイがスコード認証を使用します.\n")
else:
  print("ブラウザが開いたら Azure アカウントでログのしてください.\n")

try:
  # 環境にに従って他の認証方式使用
  if IS_REMOTE:
    # Codespaces/Remote:DeviceCodeCredential 使用
    credential = DeviceCodeCredential(tenant_id=TENANT_ID)
    print("📱 以下ステップをに従って認証してください:")
    print("  1. 以下 URLをでカラムブラウザで開きます")
    print("  2. 表示されはコードを入力します")
    print("  3. Azure アカウントでログのします\n")
  else:
    # でカラム:InteractiveBrowserCredential 使用
    credential = InteractiveBrowserCredential(tenant_id=TENANT_ID)
  
  # サブスクリプションリスト取得
  subscription_client = SubscriptionClient(credential)
  subscriptions = list(subscription_client.subscriptions.list())
  
  print("✅ Azure 認証完了!")
  print("\n" + "=" * 80)
  print("📋 使用可能なサブスクリプションリスト:")
  print("=" * 80)
  
  for i, sub in enumerate(subscriptions, 1):
    print(f"\n{i}. {sub.display_name}")
    print(f"  Subscription ID:{sub.subscription_id}")
    print(f"  状態:{sub.state}")
  
  print("\n" + "=" * 80)
  print(f"✅ 合計 {len(subscriptions)}個のサブスクリプションを見つかりました.")
  
  # デフォルトサブスクリプション情報保存 (最初の番目サブスクリプション)
  if subscriptions:
    default_sub = subscriptions[0]
    print(f"\n💡 デフォルトサブスクリプション:{default_sub.display_name}")
    print(f"  Subscription ID:{default_sub.subscription_id}")
    
    # 環境変数で保存
    os.environ["AZURE_SUBSCRIPTION_ID"] = default_sub.subscription_id
    print("\n✅ デフォルトサブスクリプション IDが環境変数に保存なりました.")
  
except Exception as e:
  print(f"\n⚠️ 認証失敗:{e}")
  print("\n💡 解決方法:")
  print("  1. テナント IDが正しいか確認")
  if IS_REMOTE:
    print("  2. デバイがスコード認証 URLをでカラムブラウザで開いたはない確認")
    print("  3. 表示されたコードを正確に入力したはない確認")
  else:
    print("  2. ブラウザでログの完了確認")
  print("  4. アカウントに該当テナントアクセス権限があるか確認")

## Resource Groupの作成

Resource GroupはAzureリソースを論理的にグループ化するコンテナです。

In [ ]:
# Resource Group 作成
!az group create \
  --name foundry-code \
  --location swedencentral

# 作成確認
!az group show \
  --name foundry-code \
  --output table

## Foundryリソースの作成

一意の名前でFoundry（AIServices）リソースを作成します。

In [ ]:
# 環境変数設定
import os
import random
import string

# 一意の名前作成
random_suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=6))
FOUNDRY_NAME = f"foundry-{random_suffix}"
PROJECT_NAME = "default-project"
RESOURCE_GROUP = "foundry-code"
LOCATION = "swedencentral"

print(f"✅ 環境変数設定")
print(f"  Foundry:{FOUNDRY_NAME}")
print(f"  Project:{PROJECT_NAME}")
print(f"  Resource Group:{RESOURCE_GROUP}")
print(f"  Location:{LOCATION}")

In [ ]:
# Step 1:Foundry Resource 作成 (AIServices)
import subprocess
import json

subscription_id = os.environ.get('AZURE_SUBSCRIPTION_ID', '')

# 🔧 修正:API バージョンを 2025-04-01-previewで変更 (allowProjectManagement サポート)
foundry_url = f"https://management.azure.com/subscriptions/{subscription_id}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.CognitiveServices/accounts/{FOUNDRY_NAME}?api-version=2025-04-01-preview"

foundry_body = json.dumps({
  "location":LOCATION,
  "kind":"AIServices",
  "sku":{"name":"S0"},
  "identity":{"type":"SystemAssigned"},
  "properties":{
    "customSubDomainName":FOUNDRY_NAME,
    "publicNetworkAccess":"Enabled",
    "allowProjectManagement":True
  }
})

print(f"📌 Step 1:Foundry Resource 作成中:{FOUNDRY_NAME}")
print("  (AIServices タイプ, API v2025-04-01-preview)")
print("  ✅ allowProjectManagement:True")
print("  💡 disableLocalAuth:設定しないない (Azure デフォルト値を適用)")

result = subprocess.run(
  ['az', 'rest', '--method', 'PUT', '--url', foundry_url, '--body', foundry_body],
  capture_output=True, text=True
)

if result.returncode == 0:
  print("✅ Foundry Resource 作成完了")
  foundry_info = json.loads(result.stdout)
  foundry_id = foundry_info.get('id', '')
  os.environ["FOUNDRY_ID"] = foundry_id
  
  # allowProjectManagement 確認
  properties = foundry_info.get('properties', {})
  allow_project = properties.get('allowProjectManagement')
  disable_local_auth = properties.get('disableLocalAuth')
  
  print(f"  Foundry ID:{foundry_id[:70]}...")
  print(f"  allowProjectManagement:{allow_project}")
  print(f"  disableLocalAuth:{disable_local_auth}")
  
  if allow_project and not disable_local_auth:
    print("  ✅ Project 作成および API Key 認証準備完了")
  else:
    print(f"  ⚠️ 設定確認必要")
else:
  print(f"⚠️ Foundry 作成失敗:{result.stderr}")

In [ ]:
# Step 2:Foundry Project 作成 (Subresource)
import subprocess
import json
import time

print(f"📌 Step 2:Foundry Project 作成中:{PROJECT_NAME}")

foundry_id = os.environ.get('FOUNDRY_ID', '')
if foundry_id:
  # Foundry Resource 作成完了待機
  time.sleep(5)
  
  # Projectを Foundryの subresourceで作成
  project_url = f"https://management.azure.com{foundry_id}/projects/{PROJECT_NAME}?api-version=2025-04-01-preview"
  project_body = json.dumps({
    "location":LOCATION,
    "identity":{"type":"SystemAssigned"},
    "properties":{
      "friendlyName":PROJECT_NAME,
      "description":f"Foundry Project:{PROJECT_NAME}"
      # disableLocalAuth 設定削除 - Azure デフォルト値確認
    }
  })
  
  print("  💡 API Key 認証:設定しないない (Azure デフォルト値を適用)")
  
  result = subprocess.run(
    ['az', 'rest', '--method', 'PUT', '--url', project_url, '--body', project_body],
    capture_output=True, text=True
  )
  
  if result.returncode == 0:
    print("✅ Foundry Project 作成完了")
    project_info = json.loads(result.stdout)
    project_id = project_info.get('id', '')
    os.environ["PROJECT_ID"] = project_id
    print(f"  Project ID:{project_id[:70]}...")
    
    # disableLocalAuth 確認
    properties = project_info.get('properties', {})
    disable_local_auth = properties.get('disableLocalAuth', True)
    if not disable_local_auth:
      print("  ✅ API Key 認証が有効化なりました")
    else:
      print("  ⚠️ API Key 認証がまだ無効化状態です")
  else:
    print(f"⚠️ Project 作成失敗:{result.stderr}")
else:
  print("⚠️ FOUNDRY_IDがありません. が前セルをまず実行してください.")

In [ ]:
# Foundry Resource および Project 確認
import subprocess

print("📋 作成されたリソース確認:\n")

result = subprocess.run(
  ['az', 'cognitiveservices', 'account', 'show',
   '--name', FOUNDRY_NAME,
   '--resource-group', RESOURCE_GROUP,
   '--query', '{Name:name, Kind:kind, Location:location, Endpoint:properties.endpoint}',
   '--output', 'table'],
  capture_output=True, text=True
)

if result.returncode == 0:
  print("Foundry Resource:")
  print(result.stdout)
  print(f"\n✅ Foundry Resourceと Projectが作成なりました!")
  print(f"💡 Portal:https://ai.azure.com")
else:
  print(f"⚠️ 確認失敗:{result.stderr}")

## API KeyとEndpointの取得

FoundryリソースのAPI KeyとEndpointを取得します。

In [ ]:
# API Key 作成 (Foundry Resource用)
import subprocess
import json

print(f"📌 Foundry API Key 作成")
print("💡 Foundry Projectは親 AIServices リソースの Keyを使用します.\n")

foundry_id = os.environ.get('FOUNDRY_ID', '')
if foundry_id:
  # 方法 1:Azure CLI コマンドで Key 取得
  print("🔑 方法 1:Azure CLIで Key 取得")
  result = subprocess.run(
    ['az', 'cognitiveservices', 'account', 'keys', 'list',
     '--name', FOUNDRY_NAME,
     '--resource-group', RESOURCE_GROUP],
    capture_output=True, text=True
  )
  
  if result.returncode == 0:
    keys = json.loads(result.stdout)
    primary_key = keys.get('key1', '')
    
    if primary_key:
      print("✅ API Key 取得完了")
      print(f"  Key1:{primary_key[:20]}...")
      os.environ["FOUNDRY_API_KEY"] = primary_key
      
      # Endpointも取得
      endpoint_result = subprocess.run(
        ['az', 'cognitiveservices', 'account', 'show',
         '--name', FOUNDRY_NAME,
         '--resource-group', RESOURCE_GROUP,
         '--query', 'properties.endpoint',
         '--output', 'tsv'],
        capture_output=True, text=True
      )
      
      if endpoint_result.returncode == 0:
        base_endpoint = endpoint_result.stdout.strip()
        # Project エンドポイント構成
        # cognitiveservices.azure.comを services.ai.azure.comで変更
        project_endpoint = base_endpoint.replace(
          ".cognitiveservices.azure.com/",
          ".services.ai.azure.com/"
        ).rstrip('/') + f"/api/projects/{PROJECT_NAME}"
        
        os.environ["FOUNDRY_ENDPOINT"] = project_endpoint
        print(f"  Base Endpoint:{base_endpoint}")
        print(f"  Project Endpoint:{project_endpoint}")
    else:
      print("⚠️ Keyを見つかりません.")
  else:
    print(f"⚠️ Key 取得失敗:{result.stderr}")
    print("\n🔑 方法 2:REST APIで Key 取得時も")
    
    # 方法 2:REST APIで時も
    key_url = f"https://management.azure.com{foundry_id}/listKeys?api-version=2025-04-01-preview"
    
    result2 = subprocess.run(
      ['az', 'rest', '--method', 'POST', '--url', key_url],
      capture_output=True, text=True
    )
    
    if result2.returncode == 0:
      keys = json.loads(result2.stdout)
      primary_key = keys.get('key1', '') or keys.get('primaryKey', '')
      
      if primary_key:
        print("✅ API Key 取得完了 (REST API)")
        print(f"  Key:{primary_key[:20]}...")
        os.environ["FOUNDRY_API_KEY"] = primary_key
      else:
        print("⚠️ Keyを見つかりません.")
    else:
      print(f"⚠️ REST APIも失敗:{result2.stderr}")
else:
  print("⚠️ FOUNDRY_IDがありません.")

print(f"\n💡 Portalで確認:https://ai.azure.com")
print(f"💡 または Azure Portalで '{FOUNDRY_NAME}' リソースの Keys and Endpoint メニュー 確認")


### Azure Portalで確認（オプション）

作成されたリソースを視覚的に確認するには:

1. [Azure Portal](https://portal.azure.com)にアクセス
2. Resource Group `foundry`を検索
3. Foundryリソースをクリックして詳細情報を確認

または[Microsoft Foundry Portal](https://ai.azure.com)でプロジェクトを確認できます。

## 環境変数の保存

次のノートブックで使用する主要な変数をファイルに保存します。

In [ ]:
# 環境変数を JSON ファイルで保存
import json

config = {
  "FOUNDRY_NAME":FOUNDRY_NAME,
  "PROJECT_NAME":PROJECT_NAME,
  "RESOURCE_GROUP":RESOURCE_GROUP,
  "LOCATION":LOCATION,
  "AZURE_SUBSCRIPTION_ID":os.environ.get("AZURE_SUBSCRIPTION_ID", ""),
  "TENANT_ID":TENANT_ID,
  "FOUNDRY_ID":os.environ.get("FOUNDRY_ID", ""),
  "PROJECT_ID":os.environ.get("PROJECT_ID", ""),
  "FOUNDRY_API_KEY":os.environ.get("FOUNDRY_API_KEY", ""),
  "FOUNDRY_ENDPOINT":os.environ.get("FOUNDRY_ENDPOINT", "")
}

config_file = ".foundry_config.json"
with open(config_file, 'w') as f:
  json.dump(config, f, indent=2)

print(f"✅ 設定ファイル保存:{config_file}")
print(f"\n📋 保存された情報:")
print(f"  Foundry:{FOUNDRY_NAME}")
print(f"  Project:{PROJECT_NAME}")
print(f"  Location:{LOCATION}")
print(f"  Endpoint:{os.environ.get('FOUNDRY_ENDPOINT', 'N/A')}")
print(f"  API Key:{'設定になる' if os.environ.get('FOUNDRY_API_KEY') else '未設定'}")
print(f"  Type:Foundry Project (AIServices + Project subresource)")
print(f"\n💡 次のノートブックでが 設定を自動でロードします.")


## 📚 追加リソース

- [Microsoft Foundry ドキュメント](https://learn.microsoft.com/en-us/azure/ai-foundry/what-is-azure-ai-foundry?view=foundry)
- [Azure Resource Manager 概要](https://learn.microsoft.com/azure/azure-resource-manager/management/overview)
- [Azure リージョンおよびが用性領域](https://learn.microsoft.com/azure/reliability/availability-zones-overview)

## 次のステップ

環境設定が完了しました！次のモジュールに進んでください:

➡️ **[02. モデルおよびデプロイ](./02-models.ipynb)**:様々な AI モデルを探索しデプロイする方法を学習します.